# Why Compose Graphs [Step 07.01 - When one flat graph stops scaling]

> **MLCourse - Agentic AI - LangGraph**

Everything you have built in modules 01 to 06 has been a *single* `StateGraph`:
one state schema, one flat set of nodes, one compile step. That works beautifully
up to maybe a dozen nodes. Past that it starts to hurt in very specific ways, and
this notebook makes those pains concrete before we fix them.

### What you'll learn

- The four concrete failure modes of a single flat graph: **state-key collisions**,
  **untestable middles**, **no reuse**, and **all-or-nothing recompiles**.
- Why "just add another node" is not a scaling strategy.
- The two composition tools LangGraph gives you: **subgraphs** (a compiled graph
  used as a node) and **`Send`** (dynamic fan-out).
- How to *decide* which parts of a workflow deserve to become a subgraph.

### Key takeaways

- A compiled LangGraph is itself a `Runnable`, so it can be a node in another graph.
- Composition is about **boundaries**: each subgraph owns its own state schema
  and can be tested, versioned and reused on its own.
- The cost of composition is the **state boundary** - mapping keys in and out.
  That boundary is where nearly every bug lives (notebook 03 is entirely about it).

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                  # environment variable access
import time                                # timing + backoff sleeps
from pathlib import Path                   # locating the track root
from dotenv import load_dotenv             # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we find the track root `03_agentic_ai`,
# then load the (gitignored) .env that lives there. Every provider-touching
# notebook in this track uses exactly this block.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

GROQ_KEY = os.getenv("GROQ_API_KEY")       # never print this value
GROQ_MODEL = "qwen/qwen3.8-27b"            # fast hosted model, generous free tier
OLLAMA_MODEL = "llama3.1:8b"               # local fallback if Groq is unavailable


def make_llm(temperature: float = 0.0, max_tokens: int = 512):
    """Return a chat model. Groq first (fast, hosted); local Ollama as fallback.

    OpenAI is never used anywhere in this course.
    """
    if GROQ_KEY:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                        temperature=temperature, max_tokens=max_tokens)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)


def safe_invoke(model, messages, retries: int = 4, pause: float = 1.5):
    """Invoke a chat model with exponential backoff on rate limits (HTTP 429).

    Groq's free tier allows roughly 8000 tokens per minute. Teaching notebooks
    fire many small calls in a row, so a retry loop is not optional here.
    """
    delay = pause
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(pause)              # pace the next call politely
            return out
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print("  [backoff] %s -- retrying in %.1fs" % (type(exc).__name__, delay))
            time.sleep(delay)
            delay *= 2                     # exponential backoff
    raise RuntimeError("unreachable")


print("Track root :", TRACK.name)
print("Provider   :", "Groq / " + GROQ_MODEL if GROQ_KEY else "Ollama / " + OLLAMA_MODEL)


### 1. A realistic flat graph

Let's build a document-processing pipeline the way you would naturally write it
after module 01: one state schema, everything at the top level.

The pipeline: **ingest -> clean -> chunk -> summarise each chunk -> score -> assemble
-> critique -> finalise**. Eight nodes. Nothing exotic. Read the state schema below
and notice how quickly the key list grows.

In [2]:
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END


class FlatState(TypedDict):
    """One schema for the WHOLE pipeline - note how many keys accumulate."""
    raw_text: str                                  # stage 1 output
    clean_text: str                                # stage 2 output
    chunks: list                                   # stage 3 output
    chunk_summaries: Annotated[list, operator.add]  # stage 4 output
    quality_score: float                           # stage 5 output
    assembled: str                                 # stage 6 output
    critique: str                                  # stage 7 output
    final: str                                     # stage 8 output
    # ... and in a real system: retry_count, errors, source_url, user_id,
    # trace_id, token_budget, model_name, cache_key, ...


print("Keys in the flat state schema:", len(FlatState.__annotations__))
for key, typ in FlatState.__annotations__.items():
    print("  -", key)

Keys in the flat state schema: 8
  - raw_text
  - clean_text
  - chunks
  - chunk_summaries
  - quality_score
  - assembled
  - critique
  - final


### The problem is already visible

Every node receives **the entire state**, including nine keys it has no business
reading. The `critique` node can accidentally read `raw_text`. The `chunk` node
can accidentally *write* `final`. Nothing stops it - Python `TypedDict` is not
enforced at runtime, and LangGraph merges whatever dict a node returns.

Let's write the nodes anyway so we can run it.

In [3]:
def ingest(state: FlatState) -> dict:
    """Stage 1: pretend to fetch a document."""
    return {"raw_text": "  LangGraph  is  a library for   stateful agent workflows.  "
                        "It   models computation as a graph of nodes and edges.  "}


def clean(state: FlatState) -> dict:
    """Stage 2: normalise whitespace."""
    return {"clean_text": " ".join(state["raw_text"].split())}


def chunk(state: FlatState) -> dict:
    """Stage 3: split into sentence-ish chunks."""
    parts = [p.strip() for p in state["clean_text"].split(".") if p.strip()]
    return {"chunks": parts}


def summarise(state: FlatState) -> dict:
    """Stage 4: one crude summary per chunk (no LLM yet - keep it deterministic)."""
    return {"chunk_summaries": [c[:40] + "..." for c in state["chunks"]]}


def score(state: FlatState) -> dict:
    """Stage 5: a fake quality heuristic."""
    return {"quality_score": round(min(1.0, len(state["chunks"]) / 3), 2)}


def assemble(state: FlatState) -> dict:
    """Stage 6: glue the summaries together."""
    return {"assembled": " | ".join(state["chunk_summaries"])}


def critique(state: FlatState) -> dict:
    """Stage 7: comment on the assembly."""
    verdict = "acceptable" if state["quality_score"] >= 0.5 else "too thin"
    return {"critique": "Coverage is %s (score=%.2f)" % (verdict, state["quality_score"])}


def finalise(state: FlatState) -> dict:
    """Stage 8: produce the deliverable."""
    return {"final": state["assembled"] + "\n[critique] " + state["critique"]}


flat = StateGraph(FlatState)
for name, fn in [("ingest", ingest), ("clean", clean), ("chunk", chunk),
                 ("summarise", summarise), ("score", score),
                 ("assemble", assemble), ("critique", critique), ("finalise", finalise)]:
    flat.add_node(name, fn)

flat.add_edge(START, "ingest")
for a, b in [("ingest", "clean"), ("clean", "chunk"), ("chunk", "summarise"),
             ("summarise", "score"), ("score", "assemble"),
             ("assemble", "critique"), ("critique", "finalise")]:
    flat.add_edge(a, b)
flat.add_edge("finalise", END)

flat_app = flat.compile()
result = flat_app.invoke({"chunk_summaries": []})
print(result["final"])

LangGraph is a library for stateful agen... | It models computation as a graph of node...
[critique] Coverage is acceptable (score=0.67)


### 2. Failure mode 1 - state-key collisions

Two stages want a key called `score`. In a flat graph there is exactly one
namespace, so the second stage silently overwrites the first. Watch it happen.

In [4]:
class CollisionState(TypedDict):
    text: str
    score: float          # who owns this key? BOTH stages think they do.


def relevance_stage(state: CollisionState) -> dict:
    """Retrieval relevance, 0-1."""
    return {"score": 0.91}


def readability_stage(state: CollisionState) -> dict:
    """Readability, 0-1. Same key name -> silent clobber."""
    return {"score": 0.42}


g = StateGraph(CollisionState)
g.add_node("relevance", relevance_stage)
g.add_node("readability", readability_stage)
g.add_edge(START, "relevance")
g.add_edge("relevance", "readability")
g.add_edge("readability", END)

out = g.compile().invoke({"text": "hello", "score": 0.0})
print("Relevance wrote 0.91, readability wrote 0.42.")
print("Final state score:", out["score"], "<- the relevance score is GONE")

Relevance wrote 0.91, readability wrote 0.42.
Final state score: 0.42 <- the relevance score is GONE


> **Pitfall:** LangGraph will not warn you. A node's return dict is merged into
> state key-by-key; without a reducer, last write wins. In a 40-node graph written
> by four people, name collisions are not hypothetical - they are inevitable.
>
> **A subgraph fixes this** because the subgraph has its *own* state schema. Its
> `score` key is private unless you explicitly map it out at the boundary.

### 3. Failure mode 2 - you cannot test the middle

Suppose the `summarise -> score -> assemble` section is the tricky part. In a flat
graph, running just that section means constructing a fake full state by hand,
and hoping you got every upstream key right.

In [5]:
# To test the middle of the flat graph you must hand-build the ENTIRE upstream state.
partial = {
    "raw_text": "",            # required by nothing here, but the schema has it
    "clean_text": "",          # ditto
    "chunks": ["a chunk", "another chunk"],
    "chunk_summaries": [],
    "quality_score": 0.0,
    "assembled": "",
    "critique": "",
    "final": "",
}
mid = summarise(partial)
partial.update(mid)
partial.update(score(partial))
partial.update(assemble(partial))
print("Manual middle-only run:", partial["assembled"])
print()
print("Notice: we had to invent 5 irrelevant keys to run 3 nodes.")
print("There is no compiled, runnable object for 'the middle'.")

Manual middle-only run: a chunk... | another chunk...

Notice: we had to invent 5 irrelevant keys to run 3 nodes.
There is no compiled, runnable object for 'the middle'.


A subgraph *is* that runnable object. `middle_app.invoke({"chunks": [...]})` -
three keys in, one key out, testable in isolation, no graph surgery.

### 4. Failure mode 3 - no reuse

If a second pipeline needs the same summarise/score/assemble logic, a flat graph
gives you two options, both bad:

1. **Copy the nodes** into the second graph - now the logic is duplicated and drifts.
2. **Merge both pipelines** into one giant graph with conditional routing - now
   the state schema is the union of both pipelines' keys and every node sees everything.

A compiled subgraph is a value. You can import it, pass it around, and drop it
into as many parents as you like.

### 5. Failure mode 4 - all-or-nothing recompiles and checkpoints

`compile()` is per-graph. Change one node's signature in a flat graph and the whole
thing recompiles; more importantly, **the checkpointer stores one blob for the whole
state**. With subgraphs, LangGraph namespaces checkpoint state per subgraph, so you
can inspect and resume at the right granularity (this connects directly to
`03_persistence_checkpointing`).

Let's make the "one blob" concrete.

In [6]:
from langgraph.checkpoint.memory import InMemorySaver

ckpt_app = flat.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "flat-demo"}}
ckpt_app.invoke({"chunk_summaries": []}, cfg)

snap = ckpt_app.get_state(cfg)
print("Checkpoint namespace :", snap.config["configurable"].get("checkpoint_ns", "(root)"))
print("Keys stored in ONE blob:", sorted(snap.values.keys()))
print()
print("Every resume, every time-travel, every state edit operates on all",
      len(snap.values), "keys at once.")

Checkpoint namespace : 
Keys stored in ONE blob: ['assembled', 'chunk_summaries', 'chunks', 'clean_text', 'critique', 'final', 'quality_score', 'raw_text']

Every resume, every time-travel, every state edit operates on all 8 keys at once.


### 6. The two composition tools

LangGraph gives you exactly two composition primitives, and they solve different problems.

| Tool | Problem it solves | Branch count | Covered in |
|------|-------------------|--------------|------------|
| **Subgraph** (compiled graph as a node) | Encapsulation, reuse, private state, isolated testing | Fixed at build time | 02, 03, 05 |
| **`Send`** (`from langgraph.types import Send`) | Fan-out where you don't know how many branches until runtime | Decided at runtime | 04, 05 |

A subgraph answers *"how do I hide complexity behind a boundary?"*
`Send` answers *"how do I run the same node N times when N is a runtime value?"*

They compose: notebook 05 uses a subgraph that internally fans out with `Send`.

### 7. When SHOULD you extract a subgraph?

Use this checklist. Extract when **two or more** are true:

- [ ] The section has a **clear input and output contract** (few keys in, few keys out).
- [ ] The section is **used more than once**, or you expect it to be.
- [ ] The section has **internal state nobody outside should see** (retry counters,
      scratch buffers, intermediate drafts).
- [ ] You want to **test or iterate on it alone**.
- [ ] The section is **owned by a different person or team**.

Do *not* extract when the section is two trivial nodes with no private state -
you would pay the boundary-mapping cost for nothing.

### 8. A one-LLM-call sanity check

Composition is a code-structure idea, not an LLM idea - but every notebook in this
module runs against a real model, so let's confirm the provider is live before
moving on to notebook 02.

In [7]:
llm = make_llm(max_tokens=180)

question = (
    "In two sentences, explain why splitting a large workflow graph into "
    "smaller reusable subgraphs makes it easier to test. Be concrete."
)
answer = safe_invoke(llm, question)
print(answer.content)
print()
print("tokens:", answer.usage_metadata)

By isolating specific logic into smaller subgraphs, you can unit test individual components in isolation using mocked inputs and outputs, rather than relying on complex, end-to-end integration tests that are brittle and slow. This modularity allows you to verify the correctness of each reusable piece independently, ensuring that changes to one subgraph do not inadvertently break unrelated parts of the larger workflow.

tokens: {'input_tokens': 37, 'output_tokens': 75, 'total_tokens': 112}


### Recap

- A flat graph fails in four specific ways: **key collisions**, **untestable middles**,
  **no reuse**, and **coarse checkpoints**.
- LangGraph's answer is **subgraphs** (fixed structure, private state) plus
  **`Send`** (runtime-determined fan-out).
- Extract a subgraph when the section has a clear contract, private state, or reuse potential.

### Next

**[02_building_a_subgraph](02_building_a_subgraph.ipynb)** - compile a graph and
drop it into a parent as a single node.